In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import pickle
import itertools



In [ ]:
from dataset_utils.paths import get_dataset_dir
from dataset_utils.utils import file_handle, grouping
from dataset_utils.utils.tree import Node

dataset_dir = get_dataset_dir()
extracted_dir = dataset_dir / "extracted"
generated_datasets_dir = dataset_dir / "datasets"


In [ ]:
extract_fol = extracted_dir
equipment_info_loc = extract_fol / "equipment_info"

In [ ]:
rule_information = file_handle.load_pickle(
    extract_fol / "TreeStruct.pkl"
)

asset_descriptions = file_handle.load_json(
    equipment_info_loc / "gre_desc.json"
)

asset_descriptions["NONE"] = {"desc":"Decription not available","detailed_desc":"NONE"}

# Observation stats 

In [ ]:
total_obs = list()
unq_obs = set()
for d in rule_information["rule_set"]:
    unq_obs = unq_obs.union(set(d["display_text"]["observations"]))
    total_obs.extend(d["display_text"]["observations"])

len(total_obs), len(unq_obs)

unq_obs = list(unq_obs)

In [ ]:
for i, unq in enumerate(unq_obs):
    print("{} : {}".format(i,unq))

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
enc = model.encode(unq_obs)

In [ ]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [ ]:
_thresh = 0.992
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

observation_map = {}

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


In [ ]:
obs_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_obs[x]) for x in r])
        major = unq_obs[r[major]]
        
        for r0 in r:
            obs_mapping[unq_obs[r0]] = major 
    else:
        obs_mapping[unq_obs[r[0]]] = unq_obs[r[0]]

# Rule stats 

In [ ]:
rule_information = file_handle.load_json(
    extracted_dir / "observations.rules.json"
)

In [ ]:
total_rules = list()
unq_rules = set()
for d in rule_information["rule_set"]: 
    s = (d["display_text"]["rules"].replace("-","").replace("OR", "").replace("(", "").replace(")", "")).split("\n")
    s = [x.strip() for x in s]
    unq_rules = unq_rules.union(set(s))
    total_rules.extend(s)

unq_rules = list(unq_rules)

In [ ]:
len(total_rules), len(unq_rules)

In [ ]:
unq_rules

In [ ]:
enc = model.encode(unq_rules)

In [ ]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [ ]:
_thresh = 0.997
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


In [ ]:
rules_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_rules[x]) for x in r])
        major = unq_rules[r[major]]

        print("Major: \n{}".format(major))
        
        for r0 in r:
            print(unq_rules[r0])
            rules_mapping[unq_rules[r0]] = major 

        print("\n")
    else:
        rules_mapping[unq_rules[r[0]]] = unq_rules[r[0]]

# Rule Grouping with semantic understanding

In [ ]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
rule_repr_dict:Dict[str,str] = {x["id"]:x["display_text"]["rules"] for x in rule_information["rule_set"]}
enc = model.encode(list(rule_repr_dict.values()))
rule_index_map = {k:v for v,k in enumerate(rule_repr_dict.keys())}
index_rule_map = {k:v for v,k in rule_index_map.items()}

norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(cosine_sim)

plt.savefig(dataset_dir / "RRSim_heatmap.png", dpi=600, bbox_inches='tight')

In [ ]:
rule_codition_similarity = np.argsort(cosine_sim, axis=-1)

rule_id_similarity_map = {}
for k,v in rule_index_map.items():
    rule_id_similarity_map[k] = [index_rule_map[x] for x in rule_codition_similarity[v]]

# Dataset Creation 

In [ ]:
extract_fol = extracted_dir
equipment_info_loc = extract_fol / "equipment_info"

random.seed(420)
np.random.seed(420)

In [ ]:
rule_information = file_handle.load_pickle(
    extract_fol / "TreeStruct.pkl"
)

asset_descriptions = file_handle.load_json(
    equipment_info_loc / "gre_desc.json"
)

In [ ]:
for  x in rule_information["rule_set"]:
    if x["id"] == "CT00001":
        print(x["display_text"]["rules"])

## Creation configuration

The cell below uses the generic configuration; unset values fall back to `DatasetCreationConfig`'s defaults.

> **DiagnosticIQ V3.1 (paper) configuration.** The released dataset was generated with `sel_random_sample = 10` (α), `elem_random_sample = 10` (β) and `n_least_sim_rules = 25` (N_sel_topk = N_eli_topk), with 4 options and one random prompt template per question. The default `elem_random_sample` is 25, so add `"elem_random_sample": 10` to reproduce the paper's composition (6,690 questions: 5,180 selection and 1,510 elimination).

In [ ]:
from dataset_utils import creation, config


dataset_config = config.DatasetCreationConfig({
    "num_options":4,
    "question_template_mode":config.QuestionTemplateMode.random,
    "question_template_random_count" : 1
})

In [ ]:
dataset = creation.create_dataset(
    rule_information,
    obs_mapping,
    {k:v["desc"] for k,v in asset_descriptions.items()},
    rule_id_similarity_map,
    dataset_config
    )

In [ ]:
len(dataset["questions"])

In [ ]:
creation.save_dataset(
    dataset,
    "DiagnosticIQ",
    generated_datasets_dir
)

In [ ]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.outputs import question_formatting, to_basic_prompt

ds = ADIQDataset(generated_datasets_dir / "DiagnosticIQ")

## Question formatting

Writes the final question text (asset description, conditions, duration and prompt) and sets `question_first`/`text_type`. In the source repository this ran as a separate notebook, `question_structuring.ipynb`, over the saved dataset.

In [ ]:
for q in ds.questions:
    question_formatting(q, ds.asset_descriptions)

ds.save(
    "DiagnosticIQ",
    generated_datasets_dir
)

In [ ]:
len(ds.questions)

In [ ]:
question_template = """
## Asset Description:
{asset_type}: {asset_description}

## Conditions:
{conditions}

## How long the conditions were met:
{temporal_condition}

{question_prompt}
{options}
"""

In [ ]:
print(to_basic_prompt(ds.questions[2000],question_template, ds.asset_descriptions))

In [ ]:
ds.questions[2000].rule_id, ds.questions[2000].correct

In [ ]:
q_type_stats = {"p":0,"n":0}
for q in ds.questions:
    if q.question_type == "positive":
        q_type_stats["p"] += 1
    else:
        q_type_stats["n"] += 1

In [ ]:
q_type_stats

In [ ]:
opt_comp = {}
for q in ds.questions:
    try:
        opt_comp[len(q.options)] += 1
    except KeyError as er:
        opt_comp[len(q.options)] = 1

In [ ]:
opt_comp